# Milestone 2 — MERFISH spatial receptor maps

Per-gene coronal, sagittal, and axial scatter plots in **CCF coordinates** (`x_ccf`, `y_ccf`, `z_ccf`).
Section coordinates (`x_section`, etc.) are retained in metadata but not used for plotting.

Genes outside the ~500-gene panel use the imputed matrix when `use_imputed_merfish: true` (~50 GB download on first access).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings

from src.config import load_config, get_figures_dir
from src.data_loaders import (
    get_abc_cache,
    load_merfish_cell_metadata,
    check_gene_availability,
    load_single_gene_merfish,
)
from src.plotting import plot_spatial, plot_family_spatial_panel

In [2]:
# Figure output directory (outside repo — edit path as needed)
OUTPUT_DIR = Path("/Users/rancze/Documents/Projects/Ach_NE_Marius_Felix/exploration")

CONFIG_PATH = PROJECT_ROOT / "receptor_query_config.yaml"
config = load_config(CONFIG_PATH)
figures_dir = get_figures_dir(config, output_dir=OUTPUT_DIR)
print(f"Figures dir: {figures_dir}")

Figures dir: /Users/rancze/Documents/GitHub/Expresso/figures


In [3]:
cache = get_abc_cache(config)
print("Manifest:", cache.current_manifest)

Manifest: releases/20260415/manifest.json


In [4]:
cell_meta = load_merfish_cell_metadata(cache, config)
print(f"MERFISH cells with CCF coords: {len(cell_meta):,}")
print("Coordinate columns:", [c for c in cell_meta.columns if "ccf" in c or "section" in c])

MERFISH cells with CCF coords: 3,739,961
Coordinate columns: ['brain_section_label', 'x_section', 'y_section', 'z_section', 'x_ccf', 'y_ccf', 'z_ccf']


In [5]:
projections = ["coronal", "sagittal", "axial"]
family_results: dict[str, dict] = {fam: {} for fam in config["_families"]}

for gene in config["_all_genes"]:
    status = check_gene_availability(cache, gene, config)
    if status == "missing":
        warnings.warn(f"{gene}: not in MERFISH panel or imputed set; skipping.")
        continue

    expr, source = load_single_gene_merfish(cache, gene, config)
    if expr is None:
        warnings.warn(f"{gene}: could not load expression; skipping.")
        continue

    family = config["_genes_flat"][gene]
    family_results[family][gene] = {"expression": expr, "source": source}

    for proj in projections:
        out = figures_dir / f"spatial_{gene}_{proj}.png"
        plot_spatial(
            cell_meta,
            expr,
            gene,
            proj,
            config,
            source_label=source,
            coord_prefix="ccf",
            save_path=out,
        )
        print(f"Saved {out} ({source})")

gene.csv: 100%|██████████| 48.4k/48.4k [00:00<00:00, 130kMB/s] 
C57BL6J-638850-log2.h5ad: 100%|██████████| 7.63G/7.63G [06:33<00:00, 19.4MMB/s]    


Saved /Users/rancze/Documents/GitHub/Expresso/figures/spatial_Htr2a_coronal.png (measured)
Saved /Users/rancze/Documents/GitHub/Expresso/figures/spatial_Htr2a_sagittal.png (measured)
Saved /Users/rancze/Documents/GitHub/Expresso/figures/spatial_Htr2a_axial.png (measured)


In [6]:
for family in config["_families"]:
    results = family_results.get(family, {})
    if not results:
        continue
    path = plot_family_spatial_panel(results, family, cell_meta, config, output_dir=OUTPUT_DIR)
    if path:
        print(f"Saved family panel {path}")

Saved family panel /Users/rancze/Documents/GitHub/Expresso/figures/spatial_panel_serotonin.png
